# Numerical oracle for the constants

Since validated computations are expensive, we first run some numerical computations to reduce the search space.
Then we will validate the constants.

In [ ]:
import Pkg;
Pkg.activate(".")
Pkg.instantiate()

In [ ]:
using Plots

We fix the truncation for the Galerkin approximation.

In [ ]:
K = 256
N = 2 * K + 1

We start by defining the dynamic
$$ 
T_c(x) = 2x+c\cdot sinpi(2x)+0.25 \quad \textrm{mod $1$}
$$
with 
$c = \frac{1}{2\pi}-\frac{1}{16}$.

In [ ]:
T(x) = 2 * x + (1 / (2 * pi) - 1 / 16) * sin(2 * π * x) + 0.25

Let $S_{r} = \{x+i y \mid -r < |y| < r\}$.

We are interested in finding $\eta$, $\rho$ such that the closure $S_{\eta}$ is contained in $T_{c}(S_{\eta})$.
We are interested in maximizing $\rho-\eta$, since it is the constant appearing in the main error term of our functional analytic treatment, i.e.:
$$
||Lf-L_Kf||_{\ell^1}\leq \left(1+2\frac{e^{-2 \pi |\rho-\alpha|}}{1-e^{-2 \pi |\rho-\alpha|}}\right)\left(e^{-2\pi K\alpha}+e^{-2\pi K(\alpha-\eta)}\right)||f||_{\infty, \alpha}.	
$$

For $\rho>0$, we denote by $\mathcal{I}$ the imaginary part of a complex number,
$$
\alpha_u(\rho):=\min_{\theta \in [0,1]}|\mathcal{I}(T_c(x+i\rho))|
$$
where $_u$ stays for upper and 
$$
UR(\rho) := \alpha_u(\rho)-\rho.
$$
We would like to maximize the function $UR$.

In [ ]:
α_u(ρ) = minimum(abs.(imag.(T.([x + im * ρ for x in 0:0.001:1]))))

function UR(ρ)
    return α_u(ρ) - ρ
end

Similarly, we would like to treat the image under the real axis; for $\rho>0$ we define 
$$
\alpha_l(\rho):=\min_{\theta \in [0,1]}|\mathcal{I}(T_c(x-i\rho))|
$$
where $_u$ stays for upper and 
$$
LR(\rho) := \alpha_u(\rho)-\rho.
$$
We would like to maximize the function $UR$.

In [ ]:
α_l(ρ) = minimum(abs.(imag.(T.([x - im * ρ for x in 0:0.001:1]))))

function LR(ρ)
    return α_l(ρ) - ρ
end

We define now 
$$
LR(\rho) = \min\{IR(\rho),OR(\rho)\}
$$
and maximise it.

In [ ]:
ER(ρ) = min(LR(ρ), UR(ρ))

In [ ]:
plot(ER, 0, 0.3)

In [ ]:
η_rad = 0:0.00001:1

bestrad, indexrad = findmax([ER(η) for η in η_rad])

η = η_rad[indexrad]

Therefore, we have found that the value maximizing $\rho-\eta$ is 

In [ ]:
η

The value of $\rho$ can be now computed as 

In [ ]:
num_η = η
num_ρ = min(α_l(num_η), α_u(num_η))
#η_s, ρ_s, ρ_s - η_s

We want to find now an $\alpha$ that minimizes the right hand side of
$$
||Lf-L_Kf||_{\ell^1}\leq \left(1+2\frac{e^{-2 \pi |\rho-\alpha|}}{1-e^{-2 \pi |\rho-\alpha|}}\right)\left(e^{-2\pi K\alpha}+e^{-2\pi K(\alpha-\eta)}\right)||f||_{\infty, \alpha}	
$$


In [ ]:
function rhs(α, K; η, ρ)
    Dρα = ρ - α
    Dαη = α - η
    coeff_1 = 1 + 2 * (exp(-2 * π * (Dρα))) / (1 - (exp(-2 * π * (Dρα))))
    coeff_2 = exp(-2 * π * K * α) + exp(-2 * π * K * (Dαη))
    return coeff_1 * coeff_2
end

In [ ]:
plot(α -> rhs(α, K; η=η, ρ=ρ_s), η_s, ρ_s)

In [ ]:
α_arr = LinRange(η_s, ρ_s, 10000)
val_min, idx = findmin(map(α -> rhs(α, K; η=η_s, ρ=ρ_s), α_arr))
α_s = α_arr[idx]
val_min, α_s

We compute now the right hand side, supposing $|\mu|>0.3$
$$
||f||_{\infty, \alpha} \leq \left(\frac{1}{|\mu|}\right)^{\frac{\alpha}{\alpha-\eta}} \left( 1+2\frac{e^{-2 \pi |\rho-\alpha|}}{1-e^{-2 \pi |\rho-\alpha|}}\right)^{\frac{\alpha}{\alpha-\eta}} ||f||_{\ell^1}.
$$


In [ ]:
function weak_strong(μ; η, α, ρ)
    s = α / (α - η)
    coeff_1 = (1 / abs(μ))^s
    coeff_2 = 1 + 2 * (exp(-2 * π * (ρ - α))) / (1 - (exp(-2 * π * (ρ - α))))^s
    return coeff_1 * coeff_2
end

In [ ]:
bws = weak_strong(0.3; η=η_s, α=α_s, ρ=ρ_s)

Clearly, the number above is ugly, but the fact that the projection error is small balances out. As a rule of thumb, the superlevel of
the resolvent that contains the eigenvalues is (remark the $sqrt(N)$ to assess the difference between $\ell^1$ and $\ell^2$ pseudospectra): 

In [ ]:
(bws * val_min) * sqrt(N)

# Certification of the constants

We certify now the constants involved in the computation above. Due to a change in interface in IntervalArithmetic.jl, we use the 0.20.9 version.

In [ ]:
Pkg.add(name = "IntervalArithmetic", version="0.20.9")

In [ ]:
using IntervalArithmetic

To guarantee that the function uses intervals, we declare a new version, we will use the prefix `I`

In [ ]:
IT(x) = 2 * x + (1 / (2 * interval(pi)) - 1 / 16) * sinpi(2 * x) + 0.25

In [ ]:
N = 1048376

Iα_u(η) = minimum(abs.(imag.(T.([interval(i, i+1)/N + im * η for i in 0:1:N-1]))))

function IUR(ρ)
    return α_u(ρ) - ρ
end

In [ ]:
Iα_l(η) = minimum(abs.(imag.(T.([interval(i, i+1)/N - im * η for i in 0:1:N-1]))))

function ILR(ρ)
    return α_l(ρ) - ρ
end

We can now certify the value of $\rho$.

In [ ]:
cert_ρ = interval(inf(hull(Iα_u(η), Iα_l(η))))

Since the image of the $\eta$ strip contains the $cert\_\rho$ strip we also fix $\eta$

In [ ]:
cert_η = interval(η)

The two values are quite near.

In [ ]:
num_ρ-cert_ρ

We compute now the right hand side, using the $\alpha$ we computed above.

In [ ]:
cert_α = interval(α_s)
rhs(cert_α, K; η = cert_η, ρ = cert_ρ)

In [ ]:
weak_strong(interval(3)/10; η = cert_η, α = cert_α, ρ = cert_ρ)

So, the necessary constants are now all certified. We go forward to certify the enclosure of the eigenvalues.

# Enclosing the eigenvalues

In [ ]:
Pkg.add("RigorousInvariantMeasures")

In [ ]:
using RigorousInvariantMeasures, BallArithmetic

In [ ]:
FourierBasis = RigorousInvariantMeasures.FourierAdjoint(K, 32768)

In [ ]:
plot(x -> mod(T(x), 1), 0, 1)

In [ ]:
savefig("2xplussome.png")

In [ ]:
P = DiscretizedOperator(FourierBasis, IT)

In [ ]:
import IntervalArithmetic
midI = IntervalArithmetic.mid
radI = IntervalArithmetic.radius

In [ ]:
midP = midI.(real.(P.L)) + im * midI.(imag.(P.L))

In [ ]:
radP = sqrt.(radI.(real.(P.L))^2 + radI.(imag.(P.L))^2)

In [ ]:
BallP = BallMatrix(midP, radP)

In [ ]:
using Pseudospectra

In [ ]:
spectralportrait(midP)

In [ ]:
dθ = 0:0.01:2π
dx, dy = [cos(x) for x in dθ], [sin(x) for x in dθ]
plot!(dx, dy, color=:green)

In [ ]:
savefig("Pseudospectra2x.png")

In [ ]:
using LinearAlgebra
abs.(diag(schur(midP).T))

With this discretization size we are not going to separate 
things that are separated by less than $0.0025$.
By inspecting the eigenvalues of the Schur matrix, we see that it may be difficult to separate the eigenvalues with norm $0.522...$, that corresponde to a double eigenvalue.
Therefore we resolve to enclose a bigger circle than $0.5$.  

In [ ]:
function certify_enclosure(enc, discr_error, weak_strong, errF, errT, norm_Z, norm_Z_inv)
    N = length(enc.points)

    r = Inf
    for i in 1:N
        abs_z = abs(Ball(enc.points[i], enc.radiuses[i]))
        r = min(r, BallArithmetic.sub_down(abs_z.c, abs_z.r))
    end
    δ = BallArithmetic.bound_resolvent(enc, errF, errT, norm_Z, norm_Z_inv)
    left_side = r #BallArithmetic.div_down(1.0, r) # @down
    right_side = BallArithmetic.mul_up(BallArithmetic.mul_up(weak_strong, discr_error), δ) #up
    @info left_side, right_side
    if left_side > right_side
        @info "The enclosure of ", enc.λ, "is certified"
        return true
    else
        return false
    end
end

In [ ]:
# A = BallP

# S = schur(Complex{Float64}.(A.c))

# bZ = BallMatrix(S.Z)
# errF = BallArithmetic.svd_bound_L2_opnorm(bZ' * bZ - I)

# bT = BallMatrix(S.T)
# errT = BallArithmetic.svd_bound_L2_opnorm(bZ * bT * bZ' - A)

# sigma_Z = BallArithmetic.svdbox(bZ)

# norm_Z = sigma_Z[1]
# norm_Z_inv = 1.0 / sigma_Z[end]

# eigv = diag(S.T)[[abs(x)>0.0001 for x in diag(S.T)]]

# errF, errT, norm_Z, norm_Z_inv

In [ ]:
eigv

In [ ]:
lambda0 = eigv[1]
lambda1 = eigv[3]

dt = LinRange(0, 1, 100)

norm_res = [1 / (svd((lambda0 + t * (lambda1 - lambda0)) * I - midP).S[end]) for t in dt]

plot(dt[2:end-1], norm_res[2:end-1])

#E = BallArithmetic._compute_exclusion_circle_level_set_priori(
#            F.T, λ, 2^(-20); rel_pearl_size = 1/128, max_initial_newton = 100)

In [ ]:
min_res_norm, i = findmin(norm_res)
@info dt[i], min_res_norm

exclusion_radius = abs(lambda0 + dt[i] * (lambda1 - lambda0))

In [ ]:
bws = weak_strong(exclusion_radius; η=η_s, α=α_s, ρ=ρ_s)

In [ ]:
U = UpperTriangular(S.T)

In [ ]:
@time svd(U)

In [ ]:
λ = eigv[end]
@info λ
ϵ = 0.000001
target = 2 * ϵ
E1 = BallArithmetic._compute_enclosure_ode(U, λ, ϵ; target=target, max_initial_newton=100, max_steps=1000)

In [ ]:
@info 1 / (sqrt(N) * BallArithmetic.bound_resolvent(E1, errF, errT, norm_Z, norm_Z_inv))
certify_enclosure(E1, val_min, bws * sqrt(N), errF, errT, norm_Z, norm_Z_inv)

In [ ]:
λ = eigv[1]
# @info λ
ϵ = (1e-9)
target = 1e-8
E2 = BallArithmetic._compute_exclusion_circle_level_set_priori(U,
    λ,
    target; N=1024, max_initial_newton=10)
#E2 = BallArithmetic._certify_circle(U, λ, target, 10)
#E2 = BallArithmetic._compute_enclosure_ode(U, λ, ϵ; target = target,  max_initial_newton = 100, max_steps = 1000)

In [ ]:
minimum([x.c for x in E2.bounds])

In [ ]:
plot(E2.points)

In [ ]:
E2ref = BallArithmetic._refine_enclosure_newton_flow(U, E2, target; rel_err=1 / 8, τ=1)

In [ ]:

E2ref = BallArithmetic._refine_enclosure_newton_flow(U, E2ref, target; rel_err=1 / 1024, τ=1)


In [ ]:
plot(E2.points)
plot!(E2ref.points)
scatter!([λ])

In [ ]:
# we refine to guarantee >ϵ
E2ref = BallArithmetic._refine_enclosure_guarantee(U, E2ref, ϵ)

In [ ]:
length(E2ref.points)

In [ ]:
minimum([x.c for x in E2ref.bounds])

In [ ]:
minimum([x.c - x.r for x in E2ref.bounds])

In [ ]:
@info 1 / (sqrt(N) * BallArithmetic.bound_resolvent(E2refref, errF, errT, norm_Z, norm_Z_inv))
certify_enclosure(E2, val_min, bws * sqrt(N), errF, errT, norm_Z, norm_Z_inv)

In [ ]:
E2.points

In [ ]:
plot(E2.points)

In [ ]:
# λ = eigvals[3]
# @info λ
# ϵ = 0.00000001
# target = 2 * ϵ
# E3 = BallArithmetic._compute_enclosure_ode(F.T, λ, ϵ; target = target,  max_initial_newton = 100, max_steps = 1000)

In [ ]:
@info 1 / (sqrt(N) * BallArithmetic.bound_resolvent(E3, errF, errT, norm_Z, norm_Z_inv))
certify_enclosure(E3, val_min, bws * sqrt(N), errF, errT, norm_Z, norm_Z_inv)

In [ ]:
E_circ = BallArithmetic._compute_central_exclusion_circle(T, exclusion_radius; max_steps=1000, rel_steps=1024)

In [ ]:
BallArithmetic.bound_resolvent(E_circ, errF, errT, norm_Z, norm_Z_inv)

In [ ]:
# using Plots

In [ ]:
# @recipe function f(::Type{BallArithmetic.Enclosure}, E::BallArithmetic.Enclosure)
#     val_sen = [sen(2*π*x) for x in 0:0.1:1]
#     val_cos = [sen(2*π*x) for x in 0:0.1:1]
    
#     out_x = []
#     out_y = []

#     for (i, z) in enumerate(E.points)
#         append!(out_x, z.+ E.radiuses[i]*val_cos)
#         append!(out_y, z.+ E.radiuses[i]*val_sen)
#     end

#     return out_x, out_y
# end

In [ ]:
# λ = eigvals[4]
# @info λ
# ϵ = 0.00001
# target = 2 * ϵ
# E = BallArithmetic._compute_enclosure_ode(F.T, λ, ϵ; target = target,  max_initial_newton = 100, max_steps = 1000)

In [ ]:
1.93 * 10.0^(36 - 47) * 9 / (0.5)

In [ ]:
bigger_radius_square(ρ) = minimum(abs.(B.([ρ * exp(im * 2 * π * θ) for θ in 0:0.001:1]))) - ρ^2

In [ ]:
smaller_radius_square(ρ) = 1 / ρ^2 - maximum(abs.(B.([exp(im * 2 * π * θ) / ρ for θ in 0:0.001:1])))

In [ ]:
radius_square(ρ) = min(bigger_radius_square(ρ), smaller_radius_square(ρ))